# <center>Procesamiento Digital de Señales de Audio</center>
### <center>Instituto de Ingeniería Eléctrica - UdelaR</center>
## <center>Filtros digitales y sus aplicaciones</center>

### Como correr este notebook

Es posible descargarlo y correrlo localmente en su computadora

Tambien pueden correrlo en Google Colab usando el siguiente link.

<table align="center">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/emidan19/audio-dsp/blob/main/practicos/P1d-filtros_digitales.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Correr en Google Colab</a>
  </td>
</table>

Este notebook reúne, en una única instancia práctica, el estudio de los filtros lineales e invariantes en el tiempo (SLITs) en tiempo discreto —su caracterización mediante respuesta al impulso, respuesta en frecuencia y diagrama de polos y ceros— junto con dos aplicaciones clásicas del procesamiento de audio: la síntesis de cuerda pulsada (algoritmo de Karplus-Strong) y un reverberador digital (estructura de Moorer).

Se puede descargar y correr el notebook de forma local.

In [ ]:
%matplotlib inline

import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches, cm

from scipy import signal
from scipy.io import wavfile

import IPython.display as ipd

plt.rcParams['figure.figsize'] = [10, 5]

## 1) Caracterización de un filtro: FIR/IIR, transformada Z y respuesta en frecuencia

Los filtros pueden tener respuesta al impulso de largo finito (FIR) o infinito (IIR). Los filtros IIR se pueden plantear como una ecuación en diferencias

$$y[n] - a_1 y[n-1] - ... - a_N y[n-N] = b_0 x[n] + b_1 x[n-1] + ... + b_M x[n-M]$$

que en el dominio de la transformada $Z$ se expresa como un cociente de polinomios

$$H(z) = \frac{Y(z)}{X(z)} =  \frac{\sum_0^M b_i z^{-i}}{1-\sum_1^N a_i z^{-i}}$$

Muchas propiedades del filtro se estudian a partir de la posición de sus polos y ceros. Cuando $H(z)$ converge sobre la circunferencia unidad ($z = e^{j\theta}$) se obtiene la respuesta en frecuencia $H(e^{j\theta})$, que es la transformada de Fourier de tiempo discreto de $h[n]$: $H(e^{j\theta}) = H(z)|_{z=e^{j\theta}}$.

Para el diagrama de polos y ceros se utiliza la siguiente función.

In [ ]:
# Copyright (c) 2011 Christopher Felton
#
# This program is free software: you can redistribute it and/or modify
# it under the terms of the GNU Lesser General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.
#
# This program is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
# GNU Lesser General Public License for more details.
#
# You should have received a copy of the GNU Lesser General Public License
# along with this program.  If not, see <http://www.gnu.org/licenses/>.
#
# The following is derived from the slides presented by
# Alexander Kain for CS506/606 "Special Topics: Speech Signal Processing"
# CSLU / OHSU, Spring Term 2011.

def zplane(b, a, filename=None):
    """Plot the complex z-plane given a transfer function."""

    ax = plt.subplot(111)

    # create the unit circle
    uc = patches.Circle((0, 0), radius=1, fill=False, color='black', ls='dashed')
    ax.add_patch(uc)

    # normalize coefficients
    b = np.asarray(b, dtype=float)
    a = np.asarray(a, dtype=float)
    kn = np.max(b) if np.max(b) > 1 else 1
    kd = np.max(a) if np.max(a) > 1 else 1
    b = b / float(kn)
    a = a / float(kd)

    # poles and zeros
    p = np.roots(a)
    z = np.roots(b)
    k = kn / float(kd)

    t1 = plt.plot(z.real, z.imag, 'go', ms=10)
    plt.setp(t1, markersize=10.0, markeredgewidth=1.0, markeredgecolor='k', markerfacecolor='g')

    t2 = plt.plot(p.real, p.imag, 'rx', ms=10)
    plt.setp(t2, markersize=12.0, markeredgewidth=3.0, markeredgecolor='r', markerfacecolor='r')

    ax.spines['left'].set_position('center')
    ax.spines['bottom'].set_position('center')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

    r = 1.5
    plt.axis('scaled')
    plt.axis([-r, r, -r, r])
    ticks = [-1, -.5, .5, 1]
    plt.xticks(ticks)
    plt.yticks(ticks)

    if filename is None:
        plt.show()
    else:
        plt.savefig(filename)

    return z, p, k


A continuación se caracteriza un filtro pasabajos Butterworth de orden 4 y frecuencia de corte $\pi/4$: su respuesta en frecuencia (en escala logarítmica), su diagrama de polos y ceros y su respuesta al impulso.

In [ ]:
orden_filtro = 4
f_corte = 0.25

b, a = signal.butter(orden_filtro, f_corte, 'low')
w, h = signal.freqz(b, a)

plt.semilogx(w, 20 * np.log10(abs(h)))
plt.title('Filtro pasabajos Butterworth: respuesta en frecuencia')
plt.xlabel('Frecuencia [radianes / segundo]')
plt.ylabel('Amplitud [dB]')
plt.margins(0, 0.1)
plt.grid(which='both', axis='both')
plt.axis([0.001, np.pi, -60, 5])
plt.show()

print('Diagrama de polos y ceros')
z, p, k = zplane(b, a)

# respuesta al impulso
butter = signal.dlti(b, a)
t, y = signal.dimpulse(butter, n=25)
plt.stem(t, np.squeeze(y))
plt.grid()
plt.title('Filtro pasabajos Butterworth: respuesta al impulso')
plt.xlabel('n [muestras]')
plt.ylabel('Amplitud')
plt.show()

La relación $Y(e^{j\theta}) = H(e^{j\theta})X(e^{j\theta})$ se puede observar en el dominio del tiempo filtrando sinusoides de distinta frecuencia. A continuación se filtran tres sinusoides de $20$, $30$ y $40~Hz$ (muestreadas a $10~kHz$) con un pasabajos de frecuencia de corte $25~Hz$.

### Ejercicio
Grafique la respuesta en frecuencia del filtro utilizado y verifique que la atenuación de cada sinusoide a la salida coincide con lo que indica dicha respuesta. Modifique el orden o la frecuencia de corte de forma que la única señal claramente atenuada sea la de $40~Hz$.

In [ ]:
fs = 10000
t = np.arange(0, 1, 1/fs)

x1 = np.cos(2 * np.pi * 20 * t)
x2 = np.cos(2 * np.pi * 30 * t)
x3 = np.cos(2 * np.pi * 40 * t)

sos = signal.butter(4, 25, 'lowpass', fs=fs, output='sos')
y1 = signal.sosfilt(sos, x1)
y2 = signal.sosfilt(sos, x2)
y3 = signal.sosfilt(sos, x3)

fig, axs = plt.subplots(3)
for ax, x, y, f in zip(axs, [x1, x2, x3], [y1, y2, y3], [20, 30, 40]):
    ax.plot(t, x)
    ax.plot(t, y)
    ax.set_title(f'Sinusoide de {f} Hz luego de un pasabajos de 25 Hz')
    ax.axis([0, 1, -2, 2])
axs[2].set_xlabel('Tiempo [segundos]')
plt.tight_layout()
plt.show()

# Graficar la respuesta en frecuencia del filtro pasabajos de 25 Hz

w, h = signal.sosfreqz(sos, worN=20000, fs=10000)

gain_db = 20 * np.log10(np.abs(h))

plt.semilogx(w, gain_db)

# Frecuencias de interés
frecuencias = np.array([20, 30, 40])
_, h_f = signal.sosfreqz(sos, worN=frecuencias, fs=10000)

ganancias_db = 20 * np.log10(np.abs(h_f))

plt.title('Filtro pasabajos Butterworth: respuesta en frecuencia')
plt.xlabel('Frecuencia [Hz]')
plt.ylabel('Amplitud [dB]')

# Marcar las tres frecuencias en la respuesta
plt.scatter(frecuencias, ganancias_db, zorder=3, color='red', label='Ganancias a 20, 30 y 40 Hz')

plt.grid(which='both', axis='both')
plt.legend()
plt.axis([1, 1000, -60, 5])
plt.show()

## 2) Fase de un filtro

La respuesta en frecuencia es, en general, una función compleja, periódica de período $2\pi$. Su notación polar (módulo y fase) muestra claramente las propiedades del sistema. Un filtro IIR (como el Butterworth) no tiene fase lineal y por lo tanto distorsiona la fase; un filtro FIR simétrico (por ejemplo una ventana de Hann) sí tiene fase lineal.

Se comparan ambos filtros pasabajos con la misma frecuencia de corte.


In [ ]:
# Pasabajos IIR, Butterworth
b, a = signal.butter(4, 100, 'lowpass', fs=10000)
w, h = signal.freqz(b, a)

# Pasabajos FIR implementado con una ventana de Hann
bFIR = np.hanning(80)
bFIR = bFIR / np.sum(bFIR)
aFIR = 1
wFIR, hFIR = signal.freqz(bFIR, aFIR)

fig, axs = plt.subplots(2)
axs[0].semilogx(w, 20 * np.log10(abs(h)), label='IIR Butterworth')
axs[0].semilogx(wFIR, 20 * np.log10(abs(hFIR)), 'r', label='FIR (Hann)')
axs[0].set_title('Módulo')
axs[0].set_ylabel('Amplitud [dB]')
axs[0].margins(0, 0.1)
axs[0].grid(which='both', axis='both')
axs[0].axis([0.001, np.pi, -60, 5])
axs[0].legend()

axs[1].plot(w, np.unwrap(np.angle(h)), label='IIR Butterworth')
axs[1].plot(wFIR, np.unwrap(np.angle(hFIR)), 'r', label='FIR (Hann)')
axs[1].set_title('Fase')
axs[1].set_ylabel('Ángulo [radianes]')
axs[1].set_xlabel('Frecuencia [radianes / segundo]')
axs[1].grid()
axs[1].axis('tight')
axs[1].legend()
plt.tight_layout()
plt.show()


La fase lineal (o lineal generalizada) preserva mejor la forma de una señal en el dominio del tiempo: todas las componentes se retardan la misma cantidad de muestras. Con fase no lineal, las distintas componentes se retardan de forma distinta y pueden aparecer distorsiones que no corresponden a un simple retardo.

Esto se observa filtrando una onda cuadrada (rica en componentes frecuenciales) con ambos filtros.


In [ ]:
t = np.arange(0, 0.5, .0001)
x = np.sign(np.cos(2 * np.pi * 10 * t))

sos = signal.butter(10, 100, 'lowpass', fs=10000, output='sos')
y = signal.sosfilt(sos, x)

sosFIR = signal.tf2sos(bFIR, aFIR)
yFIR = signal.sosfilt(sosFIR, x)

fig, axs = plt.subplots(2)
axs[0].plot(t, x)
axs[0].plot(t, y, 'g')
axs[0].set_title('Onda rectangular luego de filtro de fase NO lineal (IIR)')
axs[0].axis([0, 0.5, -1.4, 1.4])

axs[1].plot(t, x)
axs[1].plot(t, yFIR, 'g')
axs[1].set_title('Onda rectangular luego de filtro de fase lineal (FIR)')
axs[1].axis([0, 0.5, -1.4, 1.4])
axs[1].set_xlabel('Tiempo [segundos]')
plt.tight_layout()
plt.show()

### Ejercicio
A partir de la pendiente de la fase del filtro FIR, estime cuántas muestras retarda a la señal. Verifique alineando (retardando/adelantando) la entrada y la salida esa cantidad de muestras.

In [ ]:
# Tomar un tramo de la fase para calcular el retardo

delay =  ### COMPLETAR

fig, axs = plt.subplots(2)
axs[0].plot(t, x)
axs[0].plot(t, yFIR, 'g')
axs[0].set_title('Onda rectangular luego de filtro de fase lineal (FIR)')
axs[0].axis([0, 0.5, -1.4, 1.4])

axs[1].plot(t[:-delay], yFIR[delay:], 'g')
axs[1].plot(t, x, '--')
axs[1].set_title('Respuesta del FIR luego de compensar el retardo')
axs[1].axis([0, 0.5, -1.4, 1.4])
axs[1].set_xlabel('Tiempo [segundos]')
plt.tight_layout()
plt.show()

## 3) Filtros selectores de banda

El objetivo de estos filtros es dejar pasar sin alterar cierta banda de frecuencias (banda pasante) y bloquear el resto (banda atenuada), con una región intermedia (banda de transición). Hay cuatro tipos básicos: pasa-bajos, pasa-altos, pasa-banda y suprime-banda.

Parámetros que miden la calidad de estos filtros:
- **Roll-off**: ancho de la banda de transición (más angosta ⇒ roll-off más rápido).
- **Ripple en banda pasante**: oscilaciones de la magnitud dentro de la banda pasante.
- **Atenuación en banda atenuada**: qué tanto se suprimen las frecuencias fuera de la banda pasante.


In [ ]:
b1, a1 = signal.butter(4, 0.25, 'low')
b2, a2 = signal.butter(4, 0.25, 'highpass')
b3, a3 = signal.butter(6, [0.25, 0.5], 'bandpass')
b4, a4 = signal.butter(6, [0.25, 0.5], 'bandstop')

fig, axs = plt.subplots(2, 2, figsize=(11, 7))
for ax, (b, a), nombre in zip(axs.flat, [(b1, a1), (b2, a2), (b3, a3), (b4, a4)],
                               ['pasa-bajos', 'pasa-altos', 'pasa-banda', 'suprime-banda']):
    w, h = signal.freqz(b, a)
    ax.plot(w, 20 * np.log10(abs(h)))
    ax.set_title(nombre)
    ax.set_xlabel('Frecuencia [rad/s]')
    ax.set_ylabel('Amplitud [dB]')
    ax.margins(0, 0.1)
    ax.grid(which='both', axis='both')
    ax.axis([0, np.pi, -60, 5])
plt.tight_layout()
plt.show()


### Comparación entre familias de filtros IIR

Butterworth, Chebyshev tipo I, Chebyshev tipo II y elíptico ofrecen distintos compromisos entre roll-off, ripple en banda pasante y ripple/atenuación en banda atenuada, para un mismo orden.

### Ejercicio
Compare, para el mismo orden y frecuencia de corte, un filtro Butterworth, uno Chebyshev tipo I, uno Chebyshev tipo II y uno elíptico. ¿Cuál tiene el roll-off más rápido? ¿Cuál no presenta ripple en ninguna banda?


In [ ]:
orden = 6
f_corte = 0.2
ripple = 3      # dB, para Chebyshev I y elíptico
atten = 40      # dB, para Chebyshev II y elíptico

b_but, a_but = signal.butter(orden, f_corte, 'low')
b_ch1, a_ch1 = signal.cheby1(orden, ripple, f_corte, 'low')
b_ch2, a_ch2 = signal.cheby2(orden, atten, f_corte, 'low')
b_ell, a_ell = signal.ellip(orden, ripple, atten, f_corte, 'low')

plt.figure(figsize=(10, 5))
for (b, a), nombre in zip([(b_but, a_but), (b_ch1, a_ch1), (b_ch2, a_ch2), (b_ell, a_ell)],
                           ['Butterworth', 'Chebyshev I', 'Chebyshev II', 'Elíptico']):
    w, h = signal.freqz(b, a)
    plt.semilogx(w, 20 * np.log10(abs(h)), label=nombre)

plt.title(f'Comparación de filtros IIR (orden {orden}, f_corte = {f_corte}·π rad/s)')
plt.xlabel('Frecuencia [radianes / segundo]')
plt.ylabel('Amplitud [dB]')
plt.margins(0, 0.1)
plt.grid(which='both', axis='both')
plt.axis([0.001, np.pi, -80, 5])
plt.legend()
plt.show()

# Aplicaciones



Las siguientes dos secciones aplican los conceptos anteriores —en particular, el **filtro peine**— a dos problemas clásicos del procesamiento de audio: la síntesis de sonido de cuerda pulsada (algoritmo de Karplus-Strong) y un reverberador digital (estructura de Moorer).

Ambas aplicaciones se basan en el mismo filtro peine con realimentación, definido a continuación.

$$y[n] = x[n] + R^L\, y[n-L]$$

donde $L$ es el retardo en muestras y $R$ el factor de atenuación por muestra. Este filtro tiene picos en su respuesta en frecuencia espaciados cada $f_s/L$ Hz (de ahí el nombre "peine"), lo que permite usarlo como resonador a una frecuencia fundamental dada, o bien como línea de retardo con reflexiones (eco) en un reverberador.


In [ ]:
def comb_filter(x, L, R):
    """
    Filtro peine (comb filter) con realimentación.

    Parameters
    ----------
    x (numpy array) : señal de entrada
    L (int)         : retardo en muestras
    R (float)       : factor de atenuación por muestra

    Returns
    -------
    y (numpy array) : señal filtrada
    """
    N = x.size
    y = np.zeros(N)
    L = ### COMPLETAR
    g = R ** L

    
    for n in range(L):
        y[n] = ### COMPLETAR

    for n in range(L,N):
        y[n] = ### COMPLETAR

    return y

## 4) Aplicación 1: síntesis de cuerda pulsada (Karplus-Strong)

El algoritmo de Karplus y Strong sintetiza el sonido de una cuerda pulsada excitando un filtro peine con un impulso. El retardo $L$ del filtro fija la frecuencia fundamental de la nota ($f_0 = f_s/L$) y el factor $R$ controla qué tan rápido decae la energía (el tiempo de sustain de la nota).

A continuación se sintetizan las 12 notas de una octava de la escala igualmente temperada, comenzando en $A2 = 110~Hz$.

In [ ]:
def comb_filter_synth(f, fs, R, secs):
    """Sintetiza una nota excitando un filtro peine con un impulso."""
    N = int(secs * fs)
    x = np.zeros(N)
    x[0] = 1
    L = fs / f
    return comb_filter(x, L, R)

In [ ]:
fs = 22050
f_A2 = 110
# Arreglo con las 12 notas igualmente temperadas desde A2
f_scale = [f_A2 * 2 ** (n / 12) for n in range(12)] ### COMPLETAR 

tone_secs = 1
tone_samps = int(fs * tone_secs)
R = 0.999

synth_tones = np.zeros(tone_samps * len(f_scale))
for k, f in enumerate(f_scale):
    synth_tones[k * tone_samps:(k + 1) * tone_samps] = comb_filter_synth(f, fs, R, tone_secs)

t = np.linspace(0, tone_secs * len(f_scale), tone_samps * len(f_scale))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(t, synth_tones)
ax[0].set_title(f'Escala sintetizada (R={R})')
ax[0].set_xlabel('Tiempo (s)')
ax[0].set_ylabel('Amplitud')
f_spec, t_spec, Sxx = signal.spectrogram(synth_tones, fs, nfft=2048)
ax[1].pcolormesh(t_spec, f_spec, 10 * np.log10(1 + Sxx))
ax[1].set_title('Espectrograma')
ax[1].set_ylabel('Frecuencia (Hz)')
ax[1].set_xlabel('Tiempo (s)')
ax[1].set_ylim([0, 2000])
plt.tight_layout()
plt.show()

ipd.Audio(synth_tones, rate=fs)

### Ejercicio
Sintetice la misma escala con $R = 0.9$ y con $R = 0.9999$. Escuche ambos resultados y compárelos con el anterior: ¿qué efecto tiene $R$ sobre el timbre y la duración percibida de la nota?

### Filtro pasabajos en el lazo de realimentación

En una cuerda real las frecuencias altas se absorben más rápido que las bajas. Para modelar esto, Karplus y Strong agregan un filtro pasabajos (una media móvil de 2 muestras) dentro del lazo de realimentación del filtro peine.


In [ ]:
def comb_filter_lpf_synth(f, fs, R, secs, M=2):
    """Sintetiza una nota con un filtro peine que incluye un pasabajos (media móvil de M muestras) en el lazo."""
    N = int(secs * fs)
    x = np.zeros(N)
    x[0] = 1

    # Retardo inducido por frecuencia a sintetizar y media móvil M
    L =  ### COMPLETAR

    # Señal intermedia de media móvil w[n] y de salida y[n]
    y = np.zeros(N)
    w = np.zeros(N)

    # Primeras M muestras
    for n in range(M):
        y[n] = ### COMPLETAR
        w[n] = ### COMPLETAR

    # Muestras entre M y L
    for n in range(M, L):
        y[n] = ### COMPLETAR
        w[n] = ### COMPLETAR

    # Muestras de L hasta N
    for n in range(L, N):
        w[n] = ### COMPLETAR
        y[n] = ### COMPLETAR

    return y

In [ ]:
R = 0.9999
synth_tones_lpf = np.zeros(tone_samps * len(f_scale))
for k, f in enumerate(f_scale):
    synth_tones_lpf[k * tone_samps:(k + 1) * tone_samps] = comb_filter_lpf_synth(f, fs, R, tone_secs)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(t, synth_tones_lpf)
ax[0].set_title('Escala sintetizada con pasabajos en el lazo')
ax[0].set_xlabel('Tiempo (s)')
ax[0].set_ylabel('Amplitud')
f_spec, t_spec, Sxx = signal.spectrogram(synth_tones_lpf, fs, nfft=2048)
ax[1].pcolormesh(t_spec, f_spec, 10 * np.log10(1 + Sxx))
ax[1].set_title('Espectrograma')
ax[1].set_ylabel('Frecuencia (Hz)')
ax[1].set_xlabel('Tiempo (s)')
ax[1].set_ylim([0, 2000])
plt.tight_layout()
plt.show()

ipd.Audio(synth_tones_lpf, rate=fs)


### Ejercicio
Compare auditivamente esta síntesis con la anterior (sin pasabajos). ¿Qué aspecto del timbre de una cuerda pulsada real se modela mejor al incluir el pasabajos en el lazo de realimentación?


### Ejercicio

Como segunda etapa de refinamiento del modelo se incluye un filtro pasatodos (con un polo en $a$ y un cero en $1/a$) en serie con el filtro pasabajos.

Su objetivo es crear retardos fraccionarios para producir frecuencias fundamentales arbitrarias.


1. Calcular la ecuación en recurrencia del filtro peine incluyendo el pasabajos y el pasatodos en el lazo de realimentación.

1. Indicar la frecuencia fundamental de la respuesta al impulso en función de los parámetros de los filtros peine y pasatodos, y la frecuencia de muestreo.

1. Repetir el punto 2 de la parte 1 afinando el sistema en la frecuencia exacta de las notas de la escala. En el primer paso además de calcular el parámetro \(L\) del filtro peine hay que calcular el parámetro \(a\) del filtro pasatodos. 
1. Calcular experimentalmente el error (en Hz) entre la frecuencia fundamental de cada nota sintetizada y su correspondiente en la escala temperada, con y sin el uso del filtro pasatodos. ¿A qué se debe dicho error?

**Observación:** como la frecuencia fundamental de una nota no varía en el tiempo, es posible obtener una estimación bastante precisa mediante la detección del primer pico de la DFT de la señal completa de la nota.

In [ ]:
def comb_filter_lpf_apf_synth(f, fs, R, secs, a, M=2):

    N = int(secs * fs)
    x = np.zeros(N)
    x[0] = 1

    # Retardo inducido por frecuencia a sintetizar y media móvil M
    L = int(fs / f - (M - 1) / 2) ### COMPLETAR

    # Señal de salida
    y = np.copy(x)
    # Señal auxiliar salida filtrado pasatodos
    w1 = np.copy(x)
    # Señal auxiliar salida filtrado pasabajos
    w2 = np.copy(x)

    # Primera muestra de la salida
    y[0] = ### COMPLETAR

    # Ecuacion en recurrencia con n < M:
    for n in range(1,M):
        y[n] = ### COMPLETAR
    
    # Ecuacion en recurrencia con M <= n < L:
    for n in range(M,L):
        w2[n] = ### COMPLETAR
        y[n]  = ### COMPLETAR
    
    # Ecuacion en recurrencia con n >= L (x[n]= 0 con n>0):
    for n in range(L,N):
        w1[n]  = ### COMPLETAR
        w2[n]  = ### COMPLETAR
        y[n]   = ### COMPLETAR
        
    return y

## 5) Aplicación 2: reverberador de Moorer

El reverberador propuesto por Moorer [1], descrito en [2], utiliza seis filtros peine en paralelo (con distintos parámetros, modelando las reflexiones del recinto), un camino directo con ganancia $K$ (la onda directa) y un filtro pasa-todos en serie (para difundir las reflexiones y evitar coloración tonal).

![alt text](https://github.com/emidan19/audio-dsp/blob/main/notebooks/figures/moorer.png?raw=true "Moorer reverb")

.. [1] Moorer, J. A. (1979). *About this reverberation business*. Computer Music Journal, 3(2):13–28.

.. [2] Steiglitz, K. (1996). *Digital Signal Processing Primer: With Applications to Digital Audio and Computer Music.* Prentice Hall.

El filtro pasa-todos utilizado en el bucle se implementa como sigue.

In [ ]:
def all_pass(x, L, a):
    """
    Filtro pasa-todos con retardo L y ganancia a.

    Parameters
    ----------
    x (numpy array) : señal de entrada
    L (int)         : retardo en muestras
    a (float)       : ganancia del filtro

    Returns
    -------
    y (numpy array) : señal filtrada
    """
    N = x.size
    y = np.zeros(N)
    L = int(round(L))

    for n in range(N):
        x_L = x[n - L] if n >= L else 0
        y_L = y[n - L] if n >= L else 0
        y[n] = -a * x[n] + x_L + a * y_L

    return y

El reverberador combina el camino directo, los seis filtros peine en paralelo (uno por cada reflexión) y el filtro pasa-todos, con un pre-retardo $t_0$ que simula el tiempo antes de la primera reflexión.

In [ ]:
def moorer_reverb(x, fs, t0=0.05, K=1.2, delays=[0.050, 0.056, 0.061, 0.068, 0.072, 0.078], rt60=1.5):
    """
    Reverberador de Moorer.

    Parameters
    ----------
    x (numpy array)     : señal de entrada
    fs (int)            : frecuencia de muestreo en Hz
    t0 (float)          : pre-retardo en segundos
    K (float)           : ganancia del camino directo
    delays (list float) : retardos de los filtros peine en segundos
    rt60 (float)        : tiempo de reverberación (en segundos)

    Returns
    -------
    y (numpy array) : señal con reverberación
    """
    x = x.astype(float)
    ds = np.array(delays)

    # pre-retardo en muestras
    L0 = round(t0 * fs)
    # retardos de los filtros peine en muestras
    Ls = np.round(ds * fs)
    # ganancias de los filtros peine, para el rt60 pedido
    Rs = 10 ** ((-3.0 * ds) / (rt60 * fs))

    # pre-retardo
    w0 = np.append(np.zeros(L0), x)
    x_out = np.append(x, np.zeros(L0))
    ws = np.zeros(w0.shape)

    # filtros peine en paralelo
    for ind in range(len(delays)):
        ws += comb_filter(w0, Ls[ind], Rs[ind])

    # filtro pasa-todos, para la difusión de las reflexiones
    L_ap = np.round(0.005 * fs)
    g_ap = 0.7
    ap = all_pass(ws, L_ap, g_ap)

    # camino directo + reflexiones difundidas
    y = K * x_out + ap
    y = y / np.max(np.abs(y))

    return y

### Prueba del reverberador

Se descarga un audio de ejemplo y se le aplica el reverberador.

In [ ]:
import os
import urllib.request

if not os.path.exists('ohwhere.wav'):
    urllib.request.urlretrieve(
        'https://github.com/emidan19/audio-dsp/blob/main/audio/ohwhere.wav?raw=true',
        'ohwhere.wav')

fs, x = wavfile.read('ohwhere.wav')
print('Sin reverberación:')
display(ipd.Audio(x, rate=fs))

# %%
y = moorer_reverb(x, fs)
print('Con reverberación:')
display(ipd.Audio(y, rate=fs))

### Ejercicio
Pruebe distintos valores de la ganancia $K$ del camino directo y del tiempo de reverberación `rt60`. ¿Qué efecto tiene cada parámetro sobre el resultado percibido? ¿Cuál considera que es la mínima cantidad de filtros peine razonable para lograr un efecto de reverberación creíble?
